# Overview

This is notebook for ResNet model training.

## 1. Imports / Settings

In [ ]:
import os

import cv2
import timm
import torch

import numpy as np
import pandas as pd
import torch.nn as nn
import albumentations as A
import torch.optim as optim
import pyarrow.parquet as pq
import pytorch_lightning as pl
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from pyprojroot import here
from torch.mps import is_available

In [2]:
PROJECT_ROOT = here()

INPUT_PATH = "data/raw/"
IMAGES_PATH = "data/images/"
TARGET_FILE = "metadata.parquet"

OUTPUT_PATH = "output/resnet/"
LOGS_FOLDER = "logs/"
CHECKPOINTS_FOLDER = "models/"

TARGET_COLUMNS = [
    "sandstone_sludge", "siltstone_sludge", "argillite_sludge",
    "radiolarite_sludge", "coal_sludge",
    "limestone_sludge", "clay_sludge",
    "other_sludge"
]
SLUDGE_IMAGE_PATH_COLUMN = "sludge_image_path"

## 2. Classes Declaration

In [3]:
class SludgeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: str, target_cols: list, transform = None):
        self.df = df.reset_index(drop = True)
        self.img_dir = img_dir
        self.target_cols = target_cols
        self.transform = transform
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row[SLUDGE_IMAGE_PATH_COLUMN])
        if not os.path.exists(img_path):
            print(f"!!! WARN: Image (f{img_path}) doesn't exist or not available. !!!")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        targets = row[self.target_cols].values.astype(np.float32)

        if not self.transform:
            augmented = self.transform(image = image)
            image = augmented["image"]
        image = np.transpose(image, (2, 0, 1)).astype(np.float32) / 255.0

        return torch.tensor(image), torch.tensor(targets)

In [ ]:
class SludgeResNetLightning(pl.LightningModule):
    def __init__(self, model_name: str, num_classes: int, lr: float):
        super().__init__()
        self.save_hyperparameters()

        self.backbone = timm.create_model(model_name, 
                                          pretrained = True, 
                                          num_classes = 0)
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        self.softmax = nn.Softmax(dim = 1)
        self.criterion = nn.HuberLoss(delta = 1.0)
        self.mae_metric = nn.L1Loss()
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.head(features)

        return self.softmax(logits) * 100
    
    def training_step(self, batch, batch_idx):
        images, targets = batch
        preds = self(images)
        loss = self.criterion(preds, targets)

        self.log("train_loss", 
                 loss, 
                 on_step = False, 
                 on_epoch = True, 
                 prog_bar = True, 
                 logger = True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        images, targets = batch
        preds = self(images)
        loss = self.criterion(preds, targets)
        mae = self.mae_metric(preds, targets)

        self.log("val_loss", loss, on_epoch = True, prog_bar = True, logger = True)
        self.log("val_mae", mae, on_epoch = True, prog_bar = True, logger = True)

        for i, col_name in enumerate(self.trainer.datamodule.target_cols):
            class_mae = torch.mean(torch.abs(preds[:, i] - targets[:, i]))
            self.log(f"val_mae_{col_name.split("_")[0]}", class_mae, on_epoch = True, logger = True)
        return loss

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr = self.hparams.lr, weight_decay = 1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode = "min", factor = 0.5, patience = 3
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }


In [ ]:
class SludgeDataModule(pl.LightningDataModule):
    def __init__(self, df: pd.DataFrame, img_dir: str, target_cols: list, batch_size: int, num_workers: int, image_size: int, target_well_id_for_validation: int):
        super().__init__()
        self.df = df
        self.img_dir = img_dir
        self.target_cols = target_cols
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.target_well_id_for_validation = target_well_id_for_validation
        
        self.train_transform = A.Compose([
            A.RandomCrop(width = 720, height = 720, p = 1.0),
            A.Resize(width = image_size, height = image_size),
            A.HorizontalFlip(p = 0.5),
            A.VerticalFlip(p = 0.5),
            A.RandomRotate90(p = 0.5),
            A.ShiftScaleRotate(shift_limit = 0.05, scale_limit = 0.1, rotate_limit = 30, p = 0.5, border_mode = cv2.BORDER_CONSTANT),
            A.CLAHE(clip_limit = 2.0, tile_grid_size = (8, 8), p = 0.4),
            A.HueSaturationValue(hue_shift_limit = 10, sat_shift_limit = 15, val_shift_limit = 10, p = 0.4)
        ])
        
        self.val_transform = A.Compose([
            A.CenterCrop(width = 720, height = 720, p = 1.0),
            A.Resize(width = image_size, height = image_size)
        ])

    def setup(self, stage = None):
        clean_df = self.df[self.df['is_augmented'] == False].reset_index(drop = True)
        
        unique_wells = clean_df['well_id'].unique()
        val_wells = [unique_wells[self.target_well_id_for_validation]]
        
        train_df = clean_df[~clean_df['well_id'].isin(val_wells)].reset_index(drop = True)
        val_df = clean_df[clean_df['well_id'].isin(val_wells)].reset_index(drop = True)
        
        print(f"Well IDs (train): {train_df['well_id'].unique()}.")
        print(f"Well IDs (validation): {val_df['well_id'].unique()}.")
        
        self.train_dataset = SludgeDataset(train_df, self.img_dir, self.target_cols, self.train_transform)
        self.val_dataset = SludgeDataset(val_df, self.img_dir, self.target_cols, self.val_transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, 
                          batch_size = self.batch_size, 
                          shuffle = True, 
                          num_workers = self.num_workers, 
                          drop_last = True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, 
                          batch_size = self.batch_size, 
                          shuffle = False, 
                          num_workers = self.num_workers)

## 3. Training Pipeline

### 3.1. Target Variables / Constants Declaration

In [ ]:
model_training_params = {
    "model_name": "resnet50d",
    "img_size": 720,
    "batch_size": 1,
    "num_workers": 0,

    "max_epochs": 30,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "lr_factor": 0.5,
    "lr_patience": 3,

    "monitor_target": "val_loss",
    "early_stop_patience": 6,

    "target_well_for_validation_id": 1
}

MODEL_NAME = "resnet50d"
IMG_SIZE = 720
BATCH_SIZE = 1

MAX_EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LR_FACTOR = 0.5
LR_PATIENCE = 3

MONITOR_TARGET = "val_loss"
EARLY_STOP_PATIENCE = 6

NUM_WORKERS = 0

TARGET_WELL_ID_FOR_VALIDATION = 1

In [7]:
final_input_path_images = os.path.join(PROJECT_ROOT, IMAGES_PATH)
final_input_path_metadata = os.path.join(PROJECT_ROOT, INPUT_PATH, TARGET_FILE)

final_output_path = os.path.join(PROJECT_ROOT, OUTPUT_PATH)
final_output_path_logs = os.path.join(final_output_path, LOGS_FOLDER)
final_output_path_checkpoints = os.path.join(final_output_path, CHECKPOINTS_FOLDER)

os.makedirs(final_output_path_logs, exist_ok = True)
os.makedirs(final_output_path_checkpoints, exist_ok = True)

### 3.2. Training Process

In [8]:
def get_compute_engine():
    # CUDA is not available on macOS.
    # if torch.backends.cuda.is_available():
    #     compute_engine = "cuda"
    if torch.backends.mps.is_available():
        compute_engine = "mps"
    else:
        compute_engine = "cpu"
    
    return compute_engine

def run_cv():
    print("Loading the target metadata...")
    metadata_df = pd.read_parquet(final_input_path_metadata)

    data_module = SludgeDataModule(
        df = metadata_df,
        img_dir = final_input_path_images,
        target_cols = TARGET_COLUMNS,
        batch_size = BATCH_SIZE,
        num_workers = NUM_WORKERS,
        image_size = IMG_SIZE,
        target_well_id_for_validation = TARGET_WELL_ID_FOR_VALIDATION
    )
    model = SludgeResNetLightning(
        model_name = MODEL_NAME,
        num_classes = len(TARGET_COLUMNS),
        lr = LEARNING_RATE
    )
    callbacks = [
        EarlyStopping(monitor = MONITOR_TARGET,
                    patience = EARLY_STOP_PATIENCE,
                    mode = "min",
                    verbose = True
                    ),
        ModelCheckpoint(
            dirpath = final_output_path_checkpoints,
            filename = "best_resnet_sludge_epoch={epoch:02d}-val_loss={val_loss:.2f}",
            monitor = "val_loss",
            mode = "min",
            save_top_k = 1
        )
    ]
    loggers = [
        CSVLogger(save_dir = final_output_path_logs, 
                name = "csv_metrics"),
        TensorBoardLogger(save_dir = final_output_path_logs,
                        name = "tb_metrics")
    ]

    compute_engine = get_compute_engine()
    trainer = pl.Trainer(
        max_epochs = MAX_EPOCHS,
        accelerator = compute_engine,
        devices = 1,
        callbacks = callbacks,
        logger = loggers,
        log_every_n_steps = 5
    )

    trainer.fit(model, 
                datamodule = data_module)

In [9]:
print("Starting the training cycle.")

run_cv()

print("Training cycle completed.")

Starting the training cycle.
Loading the target metadata...


/Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/ml/ml-training_lab/.venv/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Well IDs (train): [4 2 3].
Well IDs (validation): [1].


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone   │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ head       │ Sequential │  1.1 M │ train │     0 │
│ 2 │ softmax    │ Softmax    │      0 │ train │     0 │
│ 3 │ criterion  │ HuberLoss  │      0 │ train │     0 │
│ 4 │ mae_metric │ L1Loss     │      0 │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 24.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.6 M                                                                                               
Total estimated model params size (MB): 98.326                                                                     
Modules in train mode: 237                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Metric val_loss improved. New best score: 11.263


Metric val_loss improved by 2.680 >= min_delta = 0.0. New best score: 8.583


Metric val_loss improved by 1.873 >= min_delta = 0.0. New best score: 6.710


Monitored metric val_loss did not improve in the last 6 records. Best score: 6.710. Signaling Trainer to stop.


Training cycle completed.
